# 用户增长分析：MySQL → Python

原始数据为 visits / orders / costs。先完成SQL导入与关联，再运行以下代码。所有输出为本次实际执行。完整公式与边界见 reports/metric_dictionary.md。Excel展示及完整项目报告见项目README。

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
root = Path.cwd()
if (root / 'analyze.py').exists(): root = root.parent
elif not (root / 'python/analyze.py').exists():
    root = root / 'outputs/user_growth_analysis'
assert (root / 'python/analyze.py').exists(), '请在项目根目录或python目录打开notebook'
sys.path.insert(0, str(root / 'python'))
from analyze import main
results = main()


{
  "users": 228169,
  "visits": 359400,
  "orders": 50414,
  "buyers": 36522,
  "conversion": 0.160065565436146,
  "revenue": 252053.78,
  "costs": 329131.62,
  "roas": 0.7658145394842342,
  "cac_visitor": 1.442490522375958,
  "cac_buyer": 9.011872843765401,
  "observed_ltv_per_visitor": 1.104680215103717,
  "observed_ltv_per_buyer": 6.901423251738677,
  "avg_dau": 905.5041095890411,
  "avg_wau_full_window": 5808.1754874651815,
  "avg_mau_full_window": 23316.157738095237,
  "mean_dau_mau_full_window": 0.040620393331867274,
  "median_purchase_delay_hours": 0.26666666666666666,
  "d1_eligible": 226742,
  "d1_returned": 5853,
  "d1_retention": 0.02581347963764984,
  "d7_eligible": 223367,
  "d7_returned": 1445,
  "d7_retention": 0.006469174049882033,
  "d30_eligible": 213204,
  "d30_returned": 536,
  "d30_retention": 0.0025140241271270707
}


## 1. 清理与关联校验

UID以字符串读取；两条异常时长置空，保留合法开始时间；窗口外订单排除。SQL和Python分别聚合收入并逐用户核对。

In [2]:
print(json.dumps(results['audit'], ensure_ascii=False, indent=2))

{
  "raw_rows": {
    "visits": 359400,
    "orders": 50415,
    "costs": 2542
  },
  "nulls": {
    "visits": {
      "row_id": 0,
      "uid": 0,
      "device": 0,
      "source_id": 0,
      "start_ts": 0,
      "end_ts": 0
    },
    "orders": {
      "row_id": 0,
      "uid": 0,
      "buy_ts": 0,
      "revenue": 0
    },
    "costs": {
      "row_id": 0,
      "source_id": 0,
      "dt": 0,
      "costs": 0
    }
  },
  "duplicates": {
    "visits": 0,
    "orders": 0,
    "costs": 0
  },
  "invalid_duration_rows": 2,
  "orders_outside_window": 1,
  "revenue_outside_window": 3.42,
  "same_time_multichannel_user_timestamp_groups": 0,
  "unmatched_orders": 0,
  "orders_before_first_visit": 0,
  "revenue_before_first_visit": 0.0,
  "low_traffic_days_below_10_dau": [
    "2018-03-31",
    "2018-04-01"
  ],
  "missing_cost_dates_by_observed_channel": {
    "1": [
      "2018-03-31",
      "2018-04-01"
    ],
    "2": [
      "2018-03-31",
      "2018-04-01"
    ],
    "3": [
      "

## 2. 新增、活跃与黏性

新增是观察期首访；WAU/MAU是滚动窗口，自然月MAU单列。起始不完整窗口不用于完整窗口均值。

In [3]:
print(results['monthly'].to_string())
print(results['daily'].tail(7).to_string())

         new_users      avg_dau  calendar_mau  new_users_mom  calendar_stickiness
month                                                                            
2017-06      13259   510.666667         13259            NaN             0.038515
2017-07      13140   532.064516         14183      -0.008975             0.037514
2017-08      10181   433.258065         11631      -0.225190             0.037250
2017-09      16704   739.066667         18975       0.640703             0.038949
2017-10      25977  1133.354839         29692       0.555136             0.038170
2017-11      27248  1328.966667         32797       0.048928             0.040521
2017-12      25268  1240.967742         31557      -0.072666             0.039325
2018-01      22624  1096.838710         28716      -0.104638             0.038196
2018-02      22197  1217.142857         28749      -0.018874             0.042337
2018-03      20589  1052.677419         27473      -0.072442             0.038317
2018-04      157

![月新增与活跃](../figures/02_monthly_growth.svg)

![用户黏性](../figures/04_stickiness.svg)

## 3. 留存与同期群

严格第1/7/30个自然日回访，不是期间任意回访；未成熟分母排除。月cohort空白是未观察。

In [4]:
print(results['retention'].tail(5).to_string(index=False))
print(results['cohort'].to_string())

cohort_date  cohort_users  d1_eligible  d1_returned  d1_retention  d7_eligible  d7_returned  d7_retention  d30_eligible  d30_returned  d30_retention
 2018-05-27           384          384         12.0      0.031250            0          NaN           NaN             0           NaN            NaN
 2018-05-28           629          629         18.0      0.028617            0          NaN           NaN             0           NaN            NaN
 2018-05-29           574          574         22.0      0.038328            0          NaN           NaN             0           NaN            NaN
 2018-05-30           899          899         92.0      0.102336            0          NaN           NaN             0           NaN            NaN
 2018-05-31          1427            0          NaN           NaN            0          NaN           NaN             0           NaN            NaN
               M0        M1        M2        M3        M4        M5        M6        M7        M8        M

![留存](../figures/05_retention.svg)

![月同期群](../figures/06_cohort.svg)

## 4. 生命周期

月末两个相邻30日窗口划分新访客、持续活跃、沉默、回流；四类互斥，合计等于已知用户。

In [5]:
print(results['lifecycle'].to_string())

         known_users current_window_start  prior_window_complete    new  continuing  dormant  returned
month                                                                                                 
2017-06        13259           2017-06-01                  False  13259           0        0         0
2017-07        26399           2017-07-02                   True  12842        1019    12516        22
2017-08        36580           2017-08-02                   True   9870         995    25271       444
2017-09        53284           2017-09-01                   True  16704        1209    34309      1062
2017-10        79261           2017-10-02                   True  25338        2126    50246      1551
2017-11       106509           2017-11-01                   True  27248        3146    73712      2403
2017-12       131777           2017-12-02                   True  24482        3463   101147      2685
2018-01       154401           2018-01-02                   True  21887  

![生命周期](../figures/07_lifecycle.svg)

## 5. 首访到首购

以1/7/30×24小时为固定窗口，只比较完整观察用户。成功购买者的首购时长不能推断全部用户。

In [6]:
print(results['conversion'].to_string(index=False))

cohort_month  users  observed_buyers  observed_conversion  eligible_1d  buyers_1d  conversion_1d  eligible_7d  buyers_7d  conversion_7d  eligible_30d  buyers_30d  conversion_30d
     2017-06  13259             2923             0.220454        13259       1804       0.136059        13259       1972       0.148729         13259        2125        0.160268
     2017-07  13140             2458             0.187062        13140       1526       0.116134        13140       1691       0.128691         13140        1805        0.137367
     2017-08  10181             1721             0.169040        10181       1097       0.107750        10181       1187       0.116590         10181        1286        0.126314
     2017-09  16704             3058             0.183070        16704       1966       0.117696        16704       2205       0.132004         16704        2441        0.146133
     2017-10  25977             4678             0.180082        25977       3302       0.127112        25977 

![固定窗口转化](../figures/08_conversion.svg)

## 6. 渠道、CAC与收入LTV

首次触点唯一归因；渠道6/7费用缺失不填0。固定30日LTV仅对整月成熟的首访月份显示。ROAS不是利润ROI。

In [7]:
print(results['channel'].to_string())
print(results['ltv30'].query('fully_mature_30d').head(18).to_string(index=False))

           new_users  buyers   orders   revenue      costs  visiting_users_any_touch  conversion  cac_visitor  cac_buyer  observed_ltv_per_visitor  observed_ltv_per_buyer      roas  marketing_surplus_rate
source_id                                                                                                                                                                                                   
1               9469    2899   5223.0  31090.55   20833.27                     18999    0.306157     2.200155   7.186364                  3.283404               10.724577  1.492351                0.492351
2              17616    3506   6392.0  46923.61   42806.04                     26245    0.199024     2.429952  12.209367                  2.663693               13.383802  1.096191                0.096191
3              66116   10473  13289.0  54511.24  141321.63                     74756    0.158403     2.137480  13.493901                  0.824479                5.204931  0.385725

![渠道CAC](../figures/09_channel_cac.svg)

![渠道ROAS](../figures/10_channel_roas.svg)

Excel展示及完整Word报告见项目README；本notebook保留MySQL与Python指标及核查结果。